# Day 12 · Exercise 5: vector_store_crud

**What you'll build:** `upsert_and_prune(collection, new_docs, ids_to_delete) -> int` — a function that upserts a batch of new or updated documents into a Chroma collection, deletes a list of stale IDs, and returns the number of documents remaining in the collection.

**Why it matters:** Real vector stores are living data — documents change and go stale. Knowing how to atomically update and retire records in a single, idempotent operation is the difference between a read-only prototype and a production knowledge base.

## Your Implementation

In [ ]:
import chromadb


def upsert_and_prune(
    collection: chromadb.Collection,
    new_docs: list[dict],
    ids_to_delete: list[str],
) -> int:
    """Upsert new or updated documents into a collection, then delete stale ones.

    Each dict in new_docs must have the keys:
        - ``id``       (str)        unique document identifier
        - ``text``     (str)        the document text to store
        - ``metadata`` (dict)       arbitrary key-value metadata

    The function:
    1. Calls ``collection.upsert()`` with the data from ``new_docs``.
    2. Calls ``collection.delete(ids=ids_to_delete)`` to remove stale records.
    3. Returns ``collection.count()`` — the number of documents that remain.

    Args:
        collection:    A Chroma ``Collection`` object (already created/opened).
        new_docs:      List of dicts, each with keys ``id``, ``text``, ``metadata``.
        ids_to_delete: List of document IDs to remove after the upsert.

    Returns:
        The total number of documents in the collection after both operations.

    Example:
        >>> import chromadb
        >>> client = chromadb.EphemeralClient()
        >>> col = client.create_collection("demo")
        >>> docs = [
        ...     {"id": "doc_1", "text": "Hello world", "metadata": {"v": 1}},
        ...     {"id": "doc_2", "text": "Goodbye world", "metadata": {"v": 1}},
        ... ]
        >>> upsert_and_prune(col, docs, [])   # 2 documents added, none deleted
        2
        >>> upsert_and_prune(col, [], ["doc_1"])  # delete doc_1
        1
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
import chromadb

_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: function is defined and callable
    try:
        assert callable(upsert_and_prune), 'upsert_and_prune is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # cannot continue — later checks would NameError

    # Shared setup: fresh in-memory collection for every sub-check
    client = chromadb.EphemeralClient()

    # Check 2: upsert adds documents; return value equals collection.count()
    try:
        col = client.create_collection('check2')
        docs = [
            {'id': 'a', 'text': 'Apple',  'metadata': {'fruit': True}},
            {'id': 'b', 'text': 'Banana', 'metadata': {'fruit': True}},
            {'id': 'c', 'text': 'Cherry', 'metadata': {'fruit': True}},
        ]
        returned = upsert_and_prune(col, docs, [])
        assert returned == 3, f'expected 3, got {returned!r}'
        assert col.count() == 3, f'collection.count() is {col.count()}, expected 3'
        print(f'{_PASS} Check 2/{total}: upsert inserts 3 docs and returns 3')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: delete removes the specified IDs; survivors are untouched
    try:
        col = client.create_collection('check3')
        docs = [
            {'id': 'x', 'text': 'X doc', 'metadata': {'_': True}},
            {'id': 'y', 'text': 'Y doc', 'metadata': {'_': True}},
            {'id': 'z', 'text': 'Z doc', 'metadata': {'_': True}},
        ]
        upsert_and_prune(col, docs, [])          # seed with 3
        remaining = upsert_and_prune(col, [], ['x', 'z'])  # delete 2
        assert remaining == 1, f'expected 1, got {remaining!r}'
        result = col.get(ids=['y'])
        assert result['documents'] == ['Y doc'], 'survivor "y" has wrong text'
        gone = col.get(ids=['x'])
        assert gone['documents'] == [], 'deleted doc "x" still present'
        print(f'{_PASS} Check 3/{total}: delete removes 2 docs; survivor intact')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: upsert-and-delete in one call; count reflects both operations
    try:
        col = client.create_collection('check4')
        initial = [
            {'id': 'p1', 'text': 'Policy v1', 'metadata': {'v': 1}},
            {'id': 'p2', 'text': 'Policy v2', 'metadata': {'v': 1}},
        ]
        upsert_and_prune(col, initial, [])  # seed: 2 docs
        # Now upsert an update to p1 + add p3, and delete p2
        update = [
            {'id': 'p1', 'text': 'Policy v1 revised', 'metadata': {'v': 2}},
            {'id': 'p3', 'text': 'Policy v3 new',     'metadata': {'v': 1}},
        ]
        count = upsert_and_prune(col, update, ['p2'])
        assert count == 2, f'expected 2, got {count!r}'
        p1 = col.get(ids=['p1'])
        assert p1['documents'] == ['Policy v1 revised'], 'p1 text not updated'
        assert p1['metadatas'][0]['v'] == 2, 'p1 metadata version not updated'
        p2 = col.get(ids=['p2'])
        assert p2['documents'] == [], 'p2 should have been deleted'
        print(f'{_PASS} Check 4/{total}: combined upsert + delete; p1 updated, p2 gone, p3 added')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))


_run_checks()

## Bonus Challenge

Right now `upsert_and_prune` stores documents without embeddings, so Chroma uses its default embedding function. Extend the function to accept an optional `embed_fn` parameter — a callable that takes a string and returns a `list[float]`. When provided, generate embeddings for each document in `new_docs` and pass them to `collection.upsert()` as the `embeddings` argument. This foreshadows Day 13, where you will swap embedding models and compare retrieval quality.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import chromadb


def upsert_and_prune(
    collection: chromadb.Collection,
    new_docs: list[dict],
    ids_to_delete: list[str],
) -> int:
    """Upsert new or updated documents into a collection, then delete stale ones."""
    if new_docs:
        ids       = [d["id"]       for d in new_docs]
        documents = [d["text"]     for d in new_docs]
        metadatas = [d["metadata"] for d in new_docs]
        kwargs: dict = {"ids": ids, "documents": documents}
        # chromadb 1.x rejects empty metadata dicts — only pass metadatas
        # when at least one document carries metadata
        if any(m for m in metadatas):
            kwargs["metadatas"] = metadatas
        collection.upsert(**kwargs)
    if ids_to_delete:
        collection.delete(ids=ids_to_delete)
    return collection.count()
```

**Why this works:** `collection.upsert()` is idempotent per ID — it inserts new records and fully overwrites existing ones, so you never get duplicates or stale embeddings. The guard `if new_docs` (and `if ids_to_delete`) prevents passing empty lists to Chroma, which some versions reject. Returning `collection.count()` after both operations gives the caller a single, authoritative snapshot of the collection size — no extra round-trips needed.
</details>
